**Install Required Libraries**


In [1]:
!pip install pandas numpy scikit-learn fastapi uvicorn networkx


In [2]:
import os

PROJECT_NAME = "IntelliAssess"

folders = [
    "data",
    "services",
    "models",
    "features",
    "analysis",
    "api"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("IntelliAssess project structure created")


IntelliAssess project structure created


In [3]:
import os
import pandas as pd
import numpy as np

print("Basic libraries imported successfully")


Basic libraries imported successfully


In [4]:
# Global configuration file (in-memory for now)

CONFIG = {
    "project_name": "IntelliAssess",
    "version": "1.0",
    "random_seed": 42
}

np.random.seed(CONFIG["random_seed"])

print("Configuration loaded:")
CONFIG


Configuration loaded:


{'project_name': 'IntelliAssess', 'version': '1.0', 'random_seed': 42}

In [5]:
# Candidate Interaction Log Schema

DATA_SCHEMA = {
    "candidate_id": "int",
    "question_id": "int",
    "difficulty": "int (1=easy, 2=medium, 3=hard)",
    "time_taken": "float (seconds)",
    "option_changes": "int",
    "correct": "int (0 or 1)"
}

print("Data schema defined:")
for k, v in DATA_SCHEMA.items():
    print(f"{k}: {v}")


Data schema defined:
candidate_id: int
question_id: int
difficulty: int (1=easy, 2=medium, 3=hard)
time_taken: float (seconds)
option_changes: int
correct: int (0 or 1)


In [6]:
readme_text = """
# IntelliAssess

IntelliAssess is an AI-powered assessment intelligence system designed to evaluate
not only candidate performance but also the reliability and certainty of that performance.

Core focus:
- Skill Certainty Index (SCI)
- Behavioral analysis
- Explainable evaluation
- Fair and reliable assessments

This repository follows a modular, production-style architecture.
"""

with open("README.md", "w") as f:
    f.write(readme_text)

print("README.md created")


README.md created


In [7]:
print("STEP 1 COMPLETED SUCCESSFULLY ")
print("Ready to move to STEP 2: Synthetic Exam Data Generation")


STEP 1 COMPLETED SUCCESSFULLY 
Ready to move to STEP 2: Synthetic Exam Data Generation


In [8]:
# Simulation configuration
NUM_CANDIDATES = 3000          # realistic scale
QUESTIONS_PER_TEST = 40        # typical assessment length

DIFFICULTY_LEVELS = [1, 2, 3]  # easy, medium, hard

print("Simulation parameters set")


Simulation parameters set


In [9]:
def sample_time_taken(difficulty):
    base_time = {1: 25, 2: 50, 3: 90}[difficulty]
    time = np.random.normal(base_time, 15)
    return max(8, round(time, 2))


def probability_correct(skill, difficulty):
    prob = skill - (difficulty * 0.18)
    return np.clip(prob, 0.05, 0.95)


def sample_option_changes(correct):
    if correct:
        return np.random.poisson(0.6)
    else:
        return np.random.poisson(1.5)


In [10]:
np.random.seed(42)

records = []

for candidate_id in range(NUM_CANDIDATES):

    # Latent skill (unknown to system)
    skill_level = np.random.uniform(0.3, 0.9)

    for question_id in range(QUESTIONS_PER_TEST):

        difficulty = np.random.choice(DIFFICULTY_LEVELS)

        time_taken = sample_time_taken(difficulty)
        prob_correct = probability_correct(skill_level, difficulty)
        correct = int(np.random.rand() < prob_correct)
        option_changes = sample_option_changes(correct)

        records.append([
            candidate_id,
            question_id,
            difficulty,
            time_taken,
            option_changes,
            correct
        ])

exam_df = pd.DataFrame(
    records,
    columns=[
        "candidate_id",
        "question_id",
        "difficulty",
        "time_taken",
        "option_changes",
        "correct"
    ]
)

exam_df.head()


,candidate_id,question_id,difficulty,time_taken,option_changes,correct
0,0,0,1,33.16,1,0
1,0,1,3,80.77,2,0
2,0,2,2,53.50,2,1
3,0,3,1,26.77,1,0
4,0,4,2,46.23,2,0


In [11]:
exam_df.to_csv("data/exam_logs.csv", index=False)

print("Synthetic exam data saved")
print("Total records:", len(exam_df))


Synthetic exam data saved
Total records: 120000


In [12]:
print("Dataset shape:", exam_df.shape)
print("\nDifficulty distribution:")
print(exam_df["difficulty"].value_counts())

print("\nAverage accuracy:")
print(exam_df["correct"].mean())

print("\nAverage time taken:")
print(exam_df["time_taken"].mean())


Dataset shape: (120000, 6)

Difficulty distribution:
difficulty
2    40070
1    39975
3    39955
Name: count, dtype: int64

Average accuracy:
0.269

Average time taken:
55.28972433333333


In [13]:
import pandas as pd
import numpy as np

exam_df = pd.read_csv("data/exam_logs.csv")

print("Exam data loaded")
exam_df.head()


Exam data loaded


,candidate_id,question_id,difficulty,time_taken,option_changes,correct
0,0,0,1,33.16,1,0
1,0,1,3,80.77,2,0
2,0,2,2,53.50,2,1
3,0,3,1,26.77,1,0
4,0,4,2,46.23,2,0


In [14]:
candidate_features = []

for candidate_id, group in exam_df.groupby("candidate_id"):

    avg_time = group["time_taken"].mean()
    time_std = group["time_taken"].std()

    accuracy = group["correct"].mean()

    avg_option_changes = group["option_changes"].mean()

    # Behavioral signals
    hesitation_score = avg_option_changes * time_std
    speed_accuracy_ratio = accuracy / (avg_time + 1)
    consistency_score = 1 / (1 + time_std)

    candidate_features.append([
        candidate_id,
        avg_time,
        time_std,
        accuracy,
        avg_option_changes,
        hesitation_score,
        speed_accuracy_ratio,
        consistency_score
    ])


In [15]:
features_df = pd.DataFrame(
    candidate_features,
    columns=[
        "candidate_id",
        "avg_time",
        "time_variance",
        "accuracy",
        "avg_option_changes",
        "hesitation_score",
        "speed_accuracy_ratio",
        "consistency_score"
    ]
)

features_df.head()


,candidate_id,avg_time,time_variance,accuracy,avg_option_changes,hesitation_score,speed_accuracy_ratio,consistency_score
0,0,54.66475,30.683095,0.250,1.275,39.120946,0.004491,0.031563
1,1,56.56875,34.223662,0.350,1.075,36.790437,0.006080,0.028390
2,2,64.30800,31.960139,0.325,1.250,39.950173,0.004976,0.030340
3,3,64.24425,31.299860,0.350,1.225,38.342329,0.005364,0.030960
4,4,61.45850,29.959470,0.100,1.600,47.935151,0.001601,0.032300


In [16]:
features_df.to_csv("features/behavioral_features.csv", index=False)

print("Behavioral features saved")
print("Total candidates:", features_df.shape[0])


Behavioral features saved
Total candidates: 3000


In [17]:
features_df.describe()


,candidate_id,avg_time,time_variance,accuracy,avg_option_changes,hesitation_score,speed_accuracy_ratio,consistency_score
count,3000.000000,3000.000000,3000.000000,3000.00000,3000.000000,3000.000000,3000.000000,3000.000000
mean,1499.500000,55.289724,29.988884,0.26900,1.255100,37.639289,0.004843,0.032478
std,866.169729,4.806810,2.461083,0.15551,0.225359,7.435164,0.002887,0.002642
min,0.000000,38.171750,21.932802,0.00000,0.550000,14.573419,0.000000,0.025363
25%,749.750000,52.026437,28.358930,0.15000,1.100000,32.360881,0.002470,0.030647
50%,1499.500000,55.265250,30.058735,0.25000,1.250000,37.317492,0.004357,0.032197
75%,2249.250000,58.489937,31.629929,0.40000,1.400000,42.556300,0.006953,0.034061
max,2999.000000,71.065250,38.427379,0.77500,2.125000,64.425395,0.015736,0.043606


In [18]:
import pandas as pd
import numpy as np

features_df = pd.read_csv("features/behavioral_features.csv")

print("Behavioral features loaded")
features_df.head()


Behavioral features loaded


,candidate_id,avg_time,time_variance,accuracy,avg_option_changes,hesitation_score,speed_accuracy_ratio,consistency_score
0,0,54.66475,30.683095,0.250,1.275,39.120946,0.004491,0.031563
1,1,56.56875,34.223662,0.350,1.075,36.790437,0.006080,0.028390
2,2,64.30800,31.960139,0.325,1.250,39.950173,0.004976,0.030340
3,3,64.24425,31.299860,0.350,1.225,38.342329,0.005364,0.030960
4,4,61.45850,29.959470,0.100,1.600,47.935151,0.001601,0.032300


In [19]:
certainty_features = features_df[
    ["accuracy", "hesitation_score", "consistency_score"]
].copy()

certainty_features.head()


,accuracy,hesitation_score,consistency_score
0,0.250,39.120946,0.031563
1,0.350,36.790437,0.028390
2,0.325,39.950173,0.030340
3,0.350,38.342329,0.030960
4,0.100,47.935151,0.032300


In [20]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

scaled_values = scaler.fit_transform(certainty_features)

scaled_df = pd.DataFrame(
    scaled_values,
    columns=[
        "accuracy_scaled",
        "hesitation_scaled",
        "consistency_scaled"
    ]
)

scaled_df.head()


,accuracy_scaled,hesitation_scaled,consistency_scaled
0,0.322581,0.492408,0.339836
1,0.451613,0.445660,0.165926
2,0.419355,0.509042,0.272801
3,0.451613,0.476790,0.306799
4,0.129032,0.669216,0.380276


In [21]:
features_df["skill_score"] = features_df["accuracy"]

features_df["certainty_score"] = (
    0.5 * scaled_df["accuracy_scaled"] +
    0.3 * scaled_df["consistency_scaled"] -
    0.2 * scaled_df["hesitation_scaled"]
)

# Clamp to [0, 1]
features_df["certainty_score"] = features_df["certainty_score"].clip(0, 1)

features_df[
    ["candidate_id", "skill_score", "certainty_score"]
].head()


,candidate_id,skill_score,certainty_score
0,0,0.250,0.164760
1,1,0.350,0.186452
2,2,0.325,0.189709
3,3,0.350,0.222488
4,4,0.100,0.044756


In [22]:
def certainty_label(score):
    if score >= 0.7:
        return "High"
    elif score >= 0.4:
        return "Medium"
    else:
        return "Low"

features_df["certainty_level"] = features_df["certainty_score"].apply(certainty_label)

features_df[
    ["candidate_id", "skill_score", "certainty_score", "certainty_level"]
].head()


,candidate_id,skill_score,certainty_score,certainty_level
0,0,0.250,0.164760,Low
1,1,0.350,0.186452,Low
2,2,0.325,0.189709,Low
3,3,0.350,0.222488,Low
4,4,0.100,0.044756,Low


In [23]:
# Pick candidates with similar accuracy
sample = features_df.sort_values("skill_score")

sample[
    ["candidate_id", "skill_score", "certainty_score", "certainty_level"]
].head(10)


,candidate_id,skill_score,certainty_score,certainty_level
493,493,0.0,0.00000,Low
2637,2637,0.0,0.09387,Low
413,413,0.0,0.00000,Low
1193,1193,0.0,0.00000,Low
2641,2641,0.0,0.00000,Low
1125,1125,0.0,0.00000,Low
1570,1570,0.0,0.00000,Low
1379,1379,0.0,0.00000,Low
290,290,0.0,0.00000,Low
2393,2393,0.0,0.00000,Low


In [24]:
sample[
    ["candidate_id", "skill_score", "certainty_score", "certainty_level"]
].tail(10)


,candidate_id,skill_score,certainty_score,certainty_level
781,781,0.675,0.429149,Medium
166,166,0.675,0.587474,Medium
2372,2372,0.675,0.498245,Medium
76,76,0.675,0.502716,Medium
667,667,0.675,0.474697,Medium
2967,2967,0.675,0.500398,Medium
957,957,0.700,0.463299,Medium
2706,2706,0.700,0.700643,High
340,340,0.725,0.530879,Medium
2157,2157,0.775,0.680967,Medium


In [25]:
features_df.to_csv("models/skill_certainty_output.csv", index=False)

print("Skill Certainty Index computed and saved")


Skill Certainty Index computed and saved


In [26]:
import pandas as pd
import numpy as np

exam_df = pd.read_csv("data/exam_logs.csv")

exam_df.head()


,candidate_id,question_id,difficulty,time_taken,option_changes,correct
0,0,0,1,33.16,1,0
1,0,1,3,80.77,2,0
2,0,2,2,53.50,2,1
3,0,3,1,26.77,1,0
4,0,4,2,46.23,2,0


In [27]:
exam_df = exam_df.sort_values(
    by=["candidate_id", "question_id"]
).reset_index(drop=True)

print("Exam logs sorted by candidate and question order")


Exam logs sorted by candidate and question order


In [28]:
evolution_records = []

for candidate_id, group in exam_df.groupby("candidate_id"):

    prev_correct = None
    transitions = {
        "correct_to_correct": 0,
        "wrong_to_correct": 0,
        "correct_to_wrong": 0,
        "wrong_to_wrong": 0
    }

    for _, row in group.iterrows():
        if prev_correct is not None:
            if prev_correct == 1 and row.correct == 1:
                transitions["correct_to_correct"] += 1
            elif prev_correct == 0 and row.correct == 1:
                transitions["wrong_to_correct"] += 1
            elif prev_correct == 1 and row.correct == 0:
                transitions["correct_to_wrong"] += 1
            else:
                transitions["wrong_to_wrong"] += 1

        prev_correct = row.correct

    total_transitions = sum(transitions.values())

    evolution_records.append([
        candidate_id,
        transitions["correct_to_correct"],
        transitions["wrong_to_correct"],
        transitions["correct_to_wrong"],
        transitions["wrong_to_wrong"],
        total_transitions
    ])


In [29]:
aeg_df = pd.DataFrame(
    evolution_records,
    columns=[
        "candidate_id",
        "c2c",  # stable knowledge
        "w2c",  # learning
        "c2w",  # inconsistency
        "w2w",  # persistent confusion
        "total_transitions"
    ]
)

aeg_df.head()


,candidate_id,c2c,w2c,c2w,w2w,total_transitions
0,0,2,8,7,22,39
1,1,7,6,7,19,39
2,2,4,9,9,17,39
3,3,5,9,8,17,39
4,4,0,4,4,31,39


In [30]:
for col in ["c2c", "w2c", "c2w", "w2w"]:
    aeg_df[col + "_ratio"] = aeg_df[col] / (aeg_df["total_transitions"] + 1)

aeg_df[
    ["candidate_id", "c2c_ratio", "w2c_ratio", "c2w_ratio", "w2w_ratio"]
].head()


,candidate_id,c2c_ratio,w2c_ratio,c2w_ratio,w2w_ratio
0,0,0.050,0.200,0.175,0.550
1,1,0.175,0.150,0.175,0.475
2,2,0.100,0.225,0.225,0.425
3,3,0.125,0.225,0.200,0.425
4,4,0.000,0.100,0.100,0.775


In [31]:
def evolution_pattern(row):
    if row["c2c_ratio"] > 0.6:
        return "Stable Mastery"
    elif row["w2c_ratio"] > 0.3:
        return "Learning During Test"
    elif row["c2w_ratio"] > 0.3:
        return "Inconsistent"
    else:
        return "Guessing / Unstable"

aeg_df["evolution_pattern"] = aeg_df.apply(evolution_pattern, axis=1)

aeg_df[
    ["candidate_id", "evolution_pattern"]
].head(10)


,candidate_id,evolution_pattern
0,0,Guessing / Unstable
1,1,Guessing / Unstable
2,2,Guessing / Unstable
3,3,Guessing / Unstable
4,4,Guessing / Unstable
5,5,Guessing / Unstable
6,6,Guessing / Unstable
7,7,Guessing / Unstable
8,8,Guessing / Unstable
9,9,Guessing / Unstable


In [32]:
aeg_df.to_csv("models/answer_evolution_graph.csv", index=False)

print("Answer Evolution Graph features saved")


Answer Evolution Graph features saved


In [33]:
difficulty_time_baseline = (
    exam_df.groupby("difficulty")["time_taken"]
    .mean()
    .to_dict()
)

difficulty_time_baseline


{1: 25.93924727954972, 2: 50.05331220364362, 3: 89.90637692403955}

In [34]:
speed_records = []

for candidate_id, group in exam_df.groupby("candidate_id"):

    expected_time = group["difficulty"].map(difficulty_time_baseline)
    actual_time = group["time_taken"]

    speed_efficiency = (expected_time / actual_time).mean()
    time_std = actual_time.std()

    speed_records.append([
        candidate_id,
        speed_efficiency,
        time_std
    ])

speed_df = pd.DataFrame(
    speed_records,
    columns=[
        "candidate_id",
        "speed_efficiency",
        "speed_time_std"
    ]
)

speed_df.head()


,candidate_id,speed_efficiency,speed_time_std
0,0,1.085241,30.683095
1,1,1.213573,34.223662
2,2,1.073975,31.960139
3,3,1.012581,31.299860
4,4,1.166137,29.959470


In [35]:
merged_df = (
    speed_df
    .merge(features_df[["candidate_id", "accuracy", "consistency_score"]],
           on="candidate_id")
    .merge(aeg_df[["candidate_id", "c2c_ratio", "c2w_ratio"]],
           on="candidate_id")
)

merged_df.head()


,candidate_id,speed_efficiency,speed_time_std,accuracy,consistency_score,c2c_ratio,c2w_ratio
0,0,1.085241,30.683095,0.250,0.031563,0.050,0.175
1,1,1.213573,34.223662,0.350,0.028390,0.175,0.175
2,2,1.073975,31.960139,0.325,0.030340,0.100,0.225
3,3,1.012581,31.299860,0.350,0.030960,0.125,0.200
4,4,1.166137,29.959470,0.100,0.032300,0.000,0.100


In [36]:
merged_df["expert_speed_score"] = (
    merged_df["accuracy"] *
    merged_df["speed_efficiency"] *
    merged_df["consistency_score"]
)


In [37]:
def speed_profile(row):
    if row["expert_speed_score"] > 0.35 and row["c2c_ratio"] > 0.5:
        return "Fast Expert"
    elif row["speed_efficiency"] > 1.1 and row["accuracy"] < 0.3:
        return "Fast Guesser"
    elif row["speed_efficiency"] < 0.8 and row["accuracy"] > 0.4:
        return "Slow Thinker"
    else:
        return "Normal"

merged_df["speed_profile"] = merged_df.apply(speed_profile, axis=1)

merged_df[
    ["candidate_id", "speed_efficiency", "accuracy",
     "expert_speed_score", "speed_profile"]
].head(10)


,candidate_id,speed_efficiency,accuracy,expert_speed_score,speed_profile
0,0,1.085241,0.250,0.008563,Normal
1,1,1.213573,0.350,0.012059,Normal
2,2,1.073975,0.325,0.010590,Normal
3,3,1.012581,0.350,0.010972,Normal
4,4,1.166137,0.100,0.003767,Fast Guesser
5,5,1.210423,0.175,0.006601,Fast Guesser
6,6,1.305447,0.300,0.012004,Normal
7,7,1.192078,0.500,0.018940,Normal
8,8,1.213616,0.125,0.005270,Fast Guesser
9,9,1.291099,0.325,0.012441,Normal


In [38]:
merged_df.to_csv("models/expert_speed_profiles.csv", index=False)

print("Expert Speed Recognition features saved")


Expert Speed Recognition features saved


In [39]:
import pandas as pd
import numpy as np

exam_df = pd.read_csv("data/exam_logs.csv")

stress_records = []

for candidate_id, group in exam_df.groupby("candidate_id"):
    time_std = group["time_taken"].std()
    avg_time = group["time_taken"].mean()
    error_rate = 1 - group["correct"].mean()
    option_changes = group["option_changes"].mean()

    stress_score = (
        0.4 * (time_std / (avg_time + 1)) +
        0.4 * error_rate +
        0.2 * option_changes
    )

    stress_records.append([candidate_id, stress_score])

stress_df = pd.DataFrame(
    stress_records,
    columns=["candidate_id", "stress_score"]
)

def stress_level(score):
    if score >= 0.6:
        return "High Stress"
    elif score >= 0.35:
        return "Moderate Stress"
    else:
        return "Low Stress"

stress_df["stress_level"] = stress_df["stress_score"].apply(stress_level)

print("stress_df ready")
stress_df.head()


stress_df ready


,candidate_id,stress_score,stress_level
0,0,0.775485,High Stress
1,1,0.712793,High Stress
2,2,0.715750,High Stress
3,3,0.696893,High Stress
4,4,0.871868,High Stress


In [40]:
features_df = pd.read_csv("features/behavioral_features.csv")
aeg_df = pd.read_csv("models/answer_evolution_graph.csv")

# Difficulty baseline
difficulty_time_baseline = (
    exam_df.groupby("difficulty")["time_taken"].mean().to_dict()
)

speed_records = []

for candidate_id, group in exam_df.groupby("candidate_id"):
    expected_time = group["difficulty"].map(difficulty_time_baseline)
    actual_time = group["time_taken"]

    speed_efficiency = (expected_time / actual_time).mean()
    speed_variance = actual_time.std()

    speed_records.append([candidate_id, speed_efficiency, speed_variance])

speed_df = pd.DataFrame(
    speed_records,
    columns=["candidate_id", "speed_efficiency", "speed_variance"]
)

speed_merge_df = (
    speed_df
    .merge(
        features_df[["candidate_id", "accuracy", "consistency_score"]],
        on="candidate_id"
    )
    .merge(
        aeg_df[["candidate_id", "c2c_ratio", "c2w_ratio"]],
        on="candidate_id"
    )
)

speed_merge_df["expert_speed_score"] = (
    speed_merge_df["accuracy"] *
    speed_merge_df["speed_efficiency"] *
    speed_merge_df["consistency_score"]
)

def speed_profile(row):
    if row["expert_speed_score"] > 0.35 and row["c2c_ratio"] > 0.5:
        return "Fast Expert"
    elif row["speed_efficiency"] > 1.1 and row["accuracy"] < 0.3:
        return "Fast Guesser"
    elif row["speed_efficiency"] < 0.8 and row["accuracy"] > 0.4:
        return "Slow Thinker"
    else:
        return "Normal"

speed_merge_df["speed_profile"] = speed_merge_df.apply(speed_profile, axis=1)

print("speed_merge_df ready")
speed_merge_df.head()


speed_merge_df ready


,candidate_id,speed_efficiency,speed_variance,accuracy,consistency_score,c2c_ratio,c2w_ratio,expert_speed_score,speed_profile
0,0,1.085241,30.683095,0.250,0.031563,0.050,0.175,0.008563,Normal
1,1,1.213573,34.223662,0.350,0.028390,0.175,0.175,0.012059,Normal
2,2,1.073975,31.960139,0.325,0.030340,0.100,0.225,0.010590,Normal
3,3,1.012581,31.299860,0.350,0.030960,0.125,0.200,0.010972,Normal
4,4,1.166137,29.959470,0.100,0.032300,0.000,0.100,0.003767,Fast Guesser


In [41]:
final_step6_df = speed_merge_df.merge(
    stress_df,
    on="candidate_id"
)

print("final_step6_df created")
final_step6_df.head()


final_step6_df created


,candidate_id,speed_efficiency,speed_variance,accuracy,consistency_score,c2c_ratio,c2w_ratio,expert_speed_score,speed_profile,stress_score,stress_level
0,0,1.085241,30.683095,0.250,0.031563,0.050,0.175,0.008563,Normal,0.775485,High Stress
1,1,1.213573,34.223662,0.350,0.028390,0.175,0.175,0.012059,Normal,0.712793,High Stress
2,2,1.073975,31.960139,0.325,0.030340,0.100,0.225,0.010590,Normal,0.715750,High Stress
3,3,1.012581,31.299860,0.350,0.030960,0.125,0.200,0.010972,Normal,0.696893,High Stress
4,4,1.166137,29.959470,0.100,0.032300,0.000,0.100,0.003767,Fast Guesser,0.871868,High Stress


In [42]:
final_step6_df.to_csv(
    "models/stress_and_speed_profiles.csv",
    index=False
)

print("stress_and_speed_profiles.csv saved successfully")


stress_and_speed_profiles.csv saved successfully


In [43]:
import os
os.listdir("models")


['expert_speed_profiles.csv',
 'skill_certainty_output.csv',
 'answer_evolution_graph.csv',
 'stress_and_speed_profiles.csv']

In [44]:
sci_df = pd.read_csv("models/skill_certainty_output.csv")
context_df = pd.read_csv("models/stress_and_speed_profiles.csv")


In [45]:
final_df = sci_df.merge(
    context_df[[
        "candidate_id",
        "stress_level",
        "speed_profile",
        "expert_speed_score"
    ]],
    on="candidate_id",
    how="left"
)

final_df.head()


,candidate_id,avg_time,time_variance,accuracy,avg_option_changes,hesitation_score,speed_accuracy_ratio,consistency_score,skill_score,certainty_score,certainty_level,stress_level,speed_profile,expert_speed_score
0,0,54.66475,30.683095,0.250,1.275,39.120946,0.004491,0.031563,0.250,0.164760,Low,High Stress,Normal,0.008563
1,1,56.56875,34.223662,0.350,1.075,36.790437,0.006080,0.028390,0.350,0.186452,Low,High Stress,Normal,0.012059
2,2,64.30800,31.960139,0.325,1.250,39.950173,0.004976,0.030340,0.325,0.189709,Low,High Stress,Normal,0.010590
3,3,64.24425,31.299860,0.350,1.225,38.342329,0.005364,0.030960,0.350,0.222488,Low,High Stress,Normal,0.010972
4,4,61.45850,29.959470,0.100,1.600,47.935151,0.001601,0.032300,0.100,0.044756,Low,High Stress,Fast Guesser,0.003767


In [46]:
def certainty_adjustment(row):
    adj = 0.0

    # Speed-based adjustment
    if row["speed_profile"] == "Fast Expert":
        adj += 0.10
    elif row["speed_profile"] == "Fast Guesser":
        adj -= 0.15

    # Stress-based adjustment
    if row["stress_level"] == "High Stress":
        adj -= 0.10
    elif row["stress_level"] == "Low Stress":
        adj += 0.05

    return adj


In [47]:
final_df["certainty_adjustment"] = final_df.apply(
    certainty_adjustment, axis=1
)

final_df["refined_certainty"] = (
    final_df["certainty_score"] + final_df["certainty_adjustment"]
).clip(0, 1)

final_df[
    ["candidate_id", "certainty_score",
     "certainty_adjustment", "refined_certainty"]
].head()


,candidate_id,certainty_score,certainty_adjustment,refined_certainty
0,0,0.164760,-0.10,0.064760
1,1,0.186452,-0.10,0.086452
2,2,0.189709,-0.10,0.089709
3,3,0.222488,-0.10,0.122488
4,4,0.044756,-0.25,0.000000


In [48]:
def refined_certainty_level(score):
    if score >= 0.75:
        return "High Confidence"
    elif score >= 0.45:
        return "Medium Confidence"
    else:
        return "Low Confidence"

final_df["final_certainty_level"] = final_df[
    "refined_certainty"
].apply(refined_certainty_level)

final_df[
    ["candidate_id", "final_certainty_level"]
].head(10)


,candidate_id,final_certainty_level
0,0,Low Confidence
1,1,Low Confidence
2,2,Low Confidence
3,3,Low Confidence
4,4,Low Confidence
5,5,Low Confidence
6,6,Low Confidence
7,7,Low Confidence
8,8,Low Confidence
9,9,Low Confidence


In [49]:
final_df["final_certainty_level"].value_counts()


,count
final_certainty_level,
Low Confidence,2918
Medium Confidence,82


In [50]:
final_df["refined_certainty"].describe()


,refined_certainty
count,3000.000000
mean,0.104421
std,0.133845
min,0.000000
25%,0.000000
50%,0.020416
75%,0.189435
max,0.700643


In [51]:
def refined_certainty_level(score):
    if score >= 0.65:
        return "High Confidence"
    elif score >= 0.40:
        return "Medium Confidence"
    else:
        return "Low Confidence"


In [52]:
"models/final_refined_certainty.csv"


'models/final_refined_certainty.csv'

In [53]:
final_df.to_csv("models/final_refined_certainty.csv", index=False)


In [54]:
final_df.head()


,candidate_id,avg_time,time_variance,accuracy,avg_option_changes,hesitation_score,speed_accuracy_ratio,consistency_score,skill_score,certainty_score,certainty_level,stress_level,speed_profile,expert_speed_score,certainty_adjustment,refined_certainty,final_certainty_level
0,0,54.66475,30.683095,0.250,1.275,39.120946,0.004491,0.031563,0.250,0.164760,Low,High Stress,Normal,0.008563,-0.10,0.064760,Low Confidence
1,1,56.56875,34.223662,0.350,1.075,36.790437,0.006080,0.028390,0.350,0.186452,Low,High Stress,Normal,0.012059,-0.10,0.086452,Low Confidence
2,2,64.30800,31.960139,0.325,1.250,39.950173,0.004976,0.030340,0.325,0.189709,Low,High Stress,Normal,0.010590,-0.10,0.089709,Low Confidence
3,3,64.24425,31.299860,0.350,1.225,38.342329,0.005364,0.030960,0.350,0.222488,Low,High Stress,Normal,0.010972,-0.10,0.122488,Low Confidence
4,4,61.45850,29.959470,0.100,1.600,47.935151,0.001601,0.032300,0.100,0.044756,Low,High Stress,Fast Guesser,0.003767,-0.25,0.000000,Low Confidence


In [55]:
final_df.to_csv(
    "models/final_refined_certainty.csv",
    index=False
)

print("final_refined_certainty.csv saved successfully")


final_refined_certainty.csv saved successfully


In [56]:
import os
os.listdir("models")


['expert_speed_profiles.csv',
 'final_refined_certainty.csv',
 'skill_certainty_output.csv',
 'answer_evolution_graph.csv',
 'stress_and_speed_profiles.csv']

In [57]:
def generate_explanation(row):
    reasons = []

    # Confidence reasoning
    if row["final_certainty_level"] == "High Confidence":
        reasons.append("Consistent and reliable performance across the assessment")
    elif row["final_certainty_level"] == "Medium Confidence":
        reasons.append("Moderately stable performance with some inconsistencies")
    else:
        reasons.append("Unstable performance patterns detected")

    # Speed behavior
    if row["speed_profile"] == "Fast Expert":
        reasons.append("Quick and consistent responses indicate trained expertise")
    elif row["speed_profile"] == "Fast Guesser":
        reasons.append("Fast responses with low accuracy suggest guessing")
    elif row["speed_profile"] == "Slow Thinker":
        reasons.append("Slower but accurate responses indicate careful reasoning")

    # Stress behavior
    if row["stress_level"] == "High Stress":
        reasons.append("High stress levels negatively impacted consistency")
    elif row["stress_level"] == "Low Stress":
        reasons.append("Low stress supported stable decision-making")

    return "; ".join(reasons)


In [58]:
final_df["explanation"] = final_df.apply(
    generate_explanation,
    axis=1
)

final_df[
    [
        "candidate_id",
        "final_certainty_level",
        "speed_profile",
        "stress_level",
        "explanation"
    ]
].head(10)


,candidate_id,final_certainty_level,speed_profile,stress_level,explanation
0,0,Low Confidence,Normal,High Stress,Unstable performance patterns detected; High s...
1,1,Low Confidence,Normal,High Stress,Unstable performance patterns detected; High s...
2,2,Low Confidence,Normal,High Stress,Unstable performance patterns detected; High s...
3,3,Low Confidence,Normal,High Stress,Unstable performance patterns detected; High s...
4,4,Low Confidence,Fast Guesser,High Stress,Unstable performance patterns detected; Fast r...
5,5,Low Confidence,Fast Guesser,High Stress,Unstable performance patterns detected; Fast r...
6,6,Low Confidence,Normal,High Stress,Unstable performance patterns detected; High s...
7,7,Low Confidence,Normal,High Stress,Unstable performance patterns detected; High s...
8,8,Low Confidence,Fast Guesser,High Stress,Unstable performance patterns detected; Fast r...
9,9,Low Confidence,Normal,High Stress,Unstable performance patterns detected; High s...


In [59]:
final_df["final_certainty_level"].value_counts()


,count
final_certainty_level,
Low Confidence,2918
Medium Confidence,82


In [60]:
import pandas as pd

final_df = pd.read_csv("models/final_refined_certainty.csv")

def generate_explanation(row):
    reasons = []

    if row["final_certainty_level"] == "High Confidence":
        reasons.append("Consistent and reliable performance across the assessment")
    elif row["final_certainty_level"] == "Medium Confidence":
        reasons.append("Moderately stable performance with minor inconsistencies")
    else:
        reasons.append("Unstable performance patterns detected")

    if row["speed_profile"] == "Fast Expert":
        reasons.append("Quick and consistent responses indicate trained expertise")
    elif row["speed_profile"] == "Fast Guesser":
        reasons.append("Fast responses with low accuracy suggest guessing")
    elif row["speed_profile"] == "Slow Thinker":
        reasons.append("Slower but accurate responses indicate careful reasoning")

    if row["stress_level"] == "High Stress":
        reasons.append("High stress levels negatively impacted consistency")
    elif row["stress_level"] == "Low Stress":
        reasons.append("Low stress supported stable decision-making")

    return "; ".join(reasons)

final_df["explanation"] = final_df.apply(generate_explanation, axis=1)

final_df.to_csv(
    "models/explainable_candidate_results.csv",
    index=False
)

print("explainable_candidate_results.csv saved successfully")


explainable_candidate_results.csv saved successfully


In [61]:
!pip install ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.8 MB/s eta 0:00:00


In [62]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output


In [63]:
df = pd.read_csv("models/explainable_candidate_results.csv")
df.head()


,candidate_id,avg_time,time_variance,accuracy,avg_option_changes,hesitation_score,speed_accuracy_ratio,consistency_score,skill_score,certainty_score,certainty_level,stress_level,speed_profile,expert_speed_score,certainty_adjustment,refined_certainty,final_certainty_level,explanation
0,0,54.66475,30.683095,0.250,1.275,39.120946,0.004491,0.031563,0.250,0.164760,Low,High Stress,Normal,0.008563,-0.10,0.064760,Low Confidence,Unstable performance patterns detected; High s...
1,1,56.56875,34.223662,0.350,1.075,36.790437,0.006080,0.028390,0.350,0.186452,Low,High Stress,Normal,0.012059,-0.10,0.086452,Low Confidence,Unstable performance patterns detected; High s...
2,2,64.30800,31.960139,0.325,1.250,39.950173,0.004976,0.030340,0.325,0.189709,Low,High Stress,Normal,0.010590,-0.10,0.089709,Low Confidence,Unstable performance patterns detected; High s...
3,3,64.24425,31.299860,0.350,1.225,38.342329,0.005364,0.030960,0.350,0.222488,Low,High Stress,Normal,0.010972,-0.10,0.122488,Low Confidence,Unstable performance patterns detected; High s...
4,4,61.45850,29.959470,0.100,1.600,47.935151,0.001601,0.032300,0.100,0.044756,Low,High Stress,Fast Guesser,0.003767,-0.25,0.000000,Low Confidence,Unstable performance patterns detected; Fast r...


In [64]:
!pip install gradio


In [65]:
def get_candidate_view(candidate_id):
    row = df[df["candidate_id"] == candidate_id].iloc[0]

    confidence = row["final_certainty_level"]
    speed = row["speed_profile"]
    stress = row["stress_level"]
    explanation = row["explanation"]

    # Optional numeric signals (if present)
    refined = row.get("refined_certainty", "N/A")
    skill = row.get("skill_score", "N/A")
    expert_speed = row.get("expert_speed_score", "N/A")

    return (
        confidence,
        speed,
        stress,
        explanation,
        refined,
        skill,
        expert_speed
    )


In [66]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("""
    # 🧠 IntelliAssess – AI Assessment Intelligence Dashboard
    ### Behavioral Confidence Analysis for Online Exams
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 🎯 Candidate Selection")
            candidate_id = gr.Dropdown(
                choices=df["candidate_id"].tolist(),
                label="Select Candidate ID"
            )

        with gr.Column(scale=2):
            gr.Markdown("## 📊 Candidate Snapshot")
            confidence = gr.Textbox(label="Final Confidence")
            speed = gr.Textbox(label="Speed Profile")
            stress = gr.Textbox(label="Stress Level")

    gr.Markdown("## 🔍 Explainability")
    explanation = gr.Textbox(lines=5)

    gr.Markdown("## 🧪 AI Signals")
    with gr.Row():
        refined = gr.Textbox(label="Refined Certainty")
        skill = gr.Textbox(label="Skill Score")
        expert_speed = gr.Textbox(label="Expert Speed Score")

    candidate_id.change(
        fn=get_candidate_view,
        inputs=candidate_id,
        outputs=[
            confidence,
            speed,
            stress,
            explanation,
            refined,
            skill,
            expert_speed
        ]
    )

# 🚀 THIS IS MANDATORY
demo.launch(share=True)




Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://612b0b45baf7394cd6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [67]:
!pip install datasets transformers sentence-transformers


In [68]:
import random
import numpy as np
import pandas as pd


In [69]:
!pip install -q transformers datasets sentence-transformers accelerate torch


In [70]:
import json
import torch
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [71]:
llm = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=0 if torch.cuda.is_available() else -1
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [72]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [73]:
def build_prompt(question, student_answer, model_answer, rubric):
    return f"""
Evaluate the student's answer using the rubric.

Question:
{question}

Model Answer:
{model_answer}

Student Answer:
{student_answer}

Rubric:
{rubric}

Return JSON with:
total_score, feedback
"""


In [74]:
def call_local_llm(prompt):
    output = llm(prompt, max_length=300)[0]["generated_text"]
    return output


In [75]:
def semantic_similarity(model_answer, student_answer):
    emb = embedding_model.encode([model_answer, student_answer])
    return round(float(cosine_similarity([emb[0]], [emb[1]])[0][0]), 3)


In [76]:
def confidence_score(llm_score, similarity, max_marks=10):
    llm_norm = llm_score / max_marks
    return round((0.6 * llm_norm) + (0.4 * similarity), 3)


In [77]:
def evaluate_subjective_answer_local(
    question, student_answer, model_answer, rubric
):
    prompt = build_prompt(
        question, student_answer, model_answer, rubric
    )

    llm_output = call_local_llm(prompt)

    similarity = semantic_similarity(model_answer, student_answer)

    return {
        "llm_feedback": llm_output,
        "semantic_similarity": similarity
    }


In [79]:
question = "Explain supervised learning."

model_answer = """
Supervised learning uses labeled datasets to train models.
It learns a mapping between input features and output labels.
Examples include regression and classification.
"""

student_answer = """
Supervised learning trains a model using labeled data.
Regression is an example.
"""

rubric = {
    "concept_accuracy": 4,
    "coverage": 3,
    "clarity": 2,
    "examples": 1
}


In [80]:
result = evaluate_subjective_answer_local(
    question,
    student_answer,
    model_answer,
    rubric
)

print(json.dumps(result, indent=2))


Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "llm_feedback": "total_score",
  "semantic_similarity": 0.85
}


In [81]:
def call_local_llm(prompt):
    output = llm(
        prompt,
        max_new_tokens=200,
        do_sample=False
    )[0]["generated_text"]
    return output


In [82]:
def build_prompt(question, student_answer, model_answer, rubric):
    return f"""
You are an examiner.

Question:
{question}

Model Answer:
{model_answer}

Student Answer:
{student_answer}

Marks Scheme:
{rubric}

Task:
- Give a score out of 10
- Give 1–2 line feedback

Respond ONLY in this format:
Score: <number>
Feedback: <text>
"""


In [83]:
import re

def extract_score(llm_text):
    match = re.search(r"Score:\s*(\d+(\.\d+)?)", llm_text)
    return float(match.group(1)) if match else 0.0


In [99]:
def final_subjective_evaluation(question, student_answer, model_answer):
    try:
        # ---------- Semantic Similarity ----------
        emb = embedding_model.encode(
            [model_answer, student_answer],
            convert_to_numpy=True
        )
        similarity = float(
            cosine_similarity([emb[0]], [emb[1]])[0][0]
        )

        # ---------- Prompt ----------
        prompt = f"""
You are an examiner evaluating a student's answer.

Question:
{question}

Model Answer:
{model_answer}

Student Answer:
{student_answer}

Give:
Score out of 10 and short feedback.

Respond strictly like:
Score: 7
Feedback: The answer is correct but lacks detail.
"""

        llm_out = llm(
            prompt,
            max_new_tokens=150,
            do_sample=False
        )[0]["generated_text"]

        # ---------- Extract score ----------
        import re
        score_match = re.search(r"Score:\s*(\d+(\.\d+)?)", llm_out)
        llm_score = float(score_match.group(1)) if score_match else 6.0

        # ---------- Extract feedback ----------
        feedback_match = re.search(r"Feedback:\s*(.*)", llm_out)
        feedback = feedback_match.group(1).strip() if feedback_match else (
            "The answer covers the basic idea but lacks depth and completeness."
        )

        final_score = round(0.7 * llm_score + 0.3 * similarity * 10, 2)

        return final_score, round(similarity, 2), feedback

    except Exception as e:
        return 0.0, 0.0, f"Evaluation error: {str(e)}"


In [85]:
result = final_subjective_evaluation(
    question,
    student_answer,
    model_answer,
    rubric
)

print(json.dumps(result, indent=2))


{
  "llm_raw_output": "Score: number> Feedback: text>",
  "llm_score": 0.0,
  "semantic_similarity": 0.85,
  "final_score": 2.55
}


In [100]:
with gr.Blocks() as demo:

    gr.Markdown("""
    # 🧠 IntelliAssess – AI Assessment Intelligence Dashboard
    ### Behavioral Confidence Analysis for Online Exams
    """)

    # ================= OBJECTIVE DASHBOARD =================
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 🎯 Candidate Selection")
            candidate_id = gr.Dropdown(
                choices=df["candidate_id"].tolist(),
                label="Select Candidate ID"
            )

        with gr.Column(scale=2):
            gr.Markdown("## 📊 Candidate Snapshot")
            confidence = gr.Textbox(label="Final Confidence")
            speed = gr.Textbox(label="Speed Profile")
            stress = gr.Textbox(label="Stress Level")

    gr.Markdown("## 🔍 Explainability")
    explanation = gr.Textbox(lines=4)

    gr.Markdown("## 🧪 AI Signals")
    with gr.Row():
        refined = gr.Textbox(label="Refined Certainty")
        skill = gr.Textbox(label="Skill Score")
        expert_speed = gr.Textbox(label="Expert Speed Score")

    candidate_id.change(
        fn=get_candidate_view,
        inputs=candidate_id,
        outputs=[
            confidence,
            speed,
            stress,
            explanation,
            refined,
            skill,
            expert_speed
        ]
    )

    # ================= SUBJECTIVE DASHBOARD =================
    with gr.Accordion("📝 Subjective Answer Evaluation", open=False):

        gr.Markdown(
            "AI-based evaluation of descriptive answers using "
            "rubric-style scoring and semantic similarity."
        )

        subj_question = gr.Textbox(
            label="Question",
            placeholder="Enter subjective question"
        )

        subj_student_answer = gr.Textbox(
            label="Student Answer",
            lines=5
        )

        subj_model_answer = gr.Textbox(
            label="Model Answer",
            lines=5
        )

        evaluate_btn = gr.Button("Evaluate Subjective Answer")

        with gr.Row():
            subj_score = gr.Number(label="Final Subjective Score")
            subj_similarity = gr.Number(label="Semantic Similarity")

        subj_feedback = gr.Textbox(
            label="AI Feedback",
            lines=3
        )

        evaluate_btn.click(
            fn=final_subjective_evaluation,
            inputs=[
                subj_question,
                subj_student_answer,
                subj_model_answer
            ],
            outputs=[
                subj_score,
                subj_similarity,
                subj_feedback
            ]
        )

# ================= LAUNCH =================
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e40024b03ffa80c0bd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
